# Landslide Planet Order Creation (Nepal)

This notebook places Planet orders for landslides listed in `landslide.csv`,
which is produced by `annotation_dedup_build.ipynb` (already resized to a fixed,
centroid-preserving `TARGET_BOX_SIZE_M` box and deduplicated).

Flow:
1. Read `landslide.csv`, which already carries per-landslide `incident_date` and
   fixed-size bounding boxes (no re-resize, no re-dedupe here).
2. Search the Planet Data API with an adaptive solar-elevation, tiered fallback ladder.
3. Detect AOI coverage by footprint point-in-polygon: if the AOI straddles the
   boundary of several scans, pack every covering scene into a single async Orders-API
   `analytic_sr_udm2` `clip` order so the downstream mosaic yields a complete composite.
4. Persist a stateful order log to Hugging Face; only landslides NOT already in the
   order log (no `order_id`) are ordered - already-ordered rows are skipped.

Notes:
- `TARGET_BOX_SIZE_M` must match the box size used by `annotation_dedup_build.ipynb`.
- Required secrets: `PLANET_API_KEY`, `HF_TOKEN`.
- Do not run this notebook and Stage 2 at the same time (shared remote logs).


In [ ]:
import os, io, json, math, re
import threading
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, upload_folder, hf_hub_download

# ---------------- inputs ----------------
LANDSLIDE_CSV  = '/kaggle/input/datasets/sanjayashrestha123/landslide/landslide.csv'   # output of annotation_dedup_build.ipynb
INCIDENT_CSV   = '/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv'



INCIDENT_ID_COL   = 'id'
INCIDENT_DATE_COL = 'incident_on'
INCIDENT_DATE_FMT = '%d-%m-%Y'     # inventory dates are DD-MM-YYYY, e.g. 20-05-2026

# ---------------- candidate box ----------------
TARGET_BOX_SIZE_M = 1000.0   # externally selected fixed box size (centroid-preserving)

# ---------------- adaptive fallback ladder ----------------
FALLBACK_TIERS = [
    {'window_days':  90, 'max_cloud': 0.05, 'sun_fraction': 0.85},
    {'window_days':  90, 'max_cloud': 0.15, 'sun_fraction': 0.85},
    {'window_days': 180, 'max_cloud': 0.15, 'sun_fraction': 0.70},
    {'window_days': 365, 'max_cloud': 0.30, 'sun_fraction': 0.60},
]

# ---------------- HF state ----------------
HF_REPO      = 'sasudo2/landslides'      # <-- CHANGE to your dataset id
HF_REPO_TYPE = 'dataset'
HF_REVISION  = 'main'
ORDER_LOG_REMOTE = 'local/order_log.json'       # persisted order state (remote file path)
ORDER_STATE_DIR  = '/kaggle/working/order_state' # local mirror folder used by batched upload_folder

LANDSLIDE_ONLY_CSV = '/kaggle/working/landslide_only.csv'

MAX_WORKERS     = 8
LOG_FLUSH_EVERY = 100   # checkpoint order log to HF after this many new entries

DATA_API   = 'https://api.planet.com/data/v1'
ORDERS_API = 'https://api.planet.com/compute/ops/orders/v2'

# ---------------- composite / coverage ----------------
# A single PSScene is only valid where real imagery exists. When an AOI straddles the
# footprint boundary of several scans, clipping a single scene leaves holes (clipped /
# incomplete imagery). Instead we submit EVERY candidate scene in ONE order so the
# downstream mosaic (Stage 2) can stitch a complete composite covering the whole AOI.
COVERAGE_THRESHOLD = 0.995      # min fraction of the AOI the chosen scenes must jointly cover
MAX_SCENES_PER_ORDER = 12       # hard cap on scenes packed into one composite order
MIN_SCENES_PER_ORDER = 3        # guarantee redundant scenes so the Stage 2 UDM-masked mosaic can
                                # fill cloud / no-data / clipped gaps even when one scene geometrically
                                # covers the whole AOI
COVERAGE_SAMPLE = 32            # sample grid (n x n) used to test AOI coverage via point-in-polygon


ModuleNotFoundError: No module named 'kaggle_secrets'

In [ ]:
secrets = UserSecretsClient()
PLANET_API_KEY = secrets.get_secret('Lokesh_planet')
HF_TOKEN       = secrets.get_secret('huggingface_token')

def make_planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    retries = Retry(total=5, backoff_factor=1.2,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=frozenset(['GET', 'POST']),
                    respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

PLANET = make_planet_session(PLANET_API_KEY)
HF_API = HfApi(token=HF_TOKEN)
print('authenticated')

In [ ]:
def bbox_to_polygon(b):
    # b: dict with min_lon, min_lat, max_lon, max_lat
    ring = [
        [b['min_lon'], b['min_lat']], [b['max_lon'], b['min_lat']],
        [b['max_lon'], b['max_lat']], [b['min_lon'], b['max_lat']],
        [b['min_lon'], b['min_lat']],
    ]
    return {'type': 'Polygon', 'coordinates': [ring]}

def iou(b1, b2):
    ix = max(0.0, min(b1['max_lon'], b2['max_lon']) - max(b1['min_lon'], b2['min_lon']))
    iy = max(0.0, min(b1['max_lat'], b2['max_lat']) - max(b1['min_lat'], b2['min_lat']))
    inter = ix * iy
    a1 = (b1['max_lon'] - b1['min_lon']) * (b1['max_lat'] - b1['min_lat'])
    a2 = (b2['max_lon'] - b2['min_lon']) * (b2['max_lat'] - b2['min_lat'])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0

def theoretical_max_elevation(incident_date, lat):
    # Adaptive solar check. For Nepal (~26-31N) max elevation varies seasonally from
    # ~85 deg (summer) down to ~38 deg (winter); a fixed absolute threshold fails in winter.
    doy  = incident_date.timetuple().tm_yday
    decl = 23.45 * math.sin(math.radians(360.0 / 365.0 * (doy - 81)))
    return 90.0 - abs(lat - decl)

In [ ]:
def quick_search(bbox, start, end, max_cloud, min_sun):
    # Proven Planet quick-search format (see planet_order_creation).
    and_filter = {
        'type': 'AndFilter',
        'config': [
            {'type': 'GeometryFilter', 'field_name': 'geometry', 'config': bbox_to_polygon(bbox)},
            {'type': 'DateRangeFilter', 'field_name': 'acquired',
             'config': {'gte': start, 'lte': end}},
            {'type': 'RangeFilter', 'field_name': 'cloud_cover', 'config': {'lte': max_cloud}},
        ],
    }
    body = {'item_types': ['PSScene'], 'filter': and_filter}
    r = PLANET.post(f'{DATA_API}/quick-search', json=body, timeout=120)
    r.raise_for_status()
    return r.json().get('features', [])

def _point_in_geometry(x, y, geom):
    # Ray-casting point-in-polygon, dependency-free (no shapely required). Handles the
    # Polygon and MultiPolygon geometries returned by the Planet quick-search API.
    gtype = geom.get('type')
    coords = geom.get('coordinates')
    if not coords:
        return False
    if gtype == 'MultiPolygon':
        rings = [poly[0] for poly in coords]
    elif gtype == 'Polygon':
        rings = [coords[0]]
    else:
        return False
    for ring in rings:
        inside = False
        j = len(ring) - 1
        for i in range(len(ring)):
            xi, yi = ring[i][0], ring[i][1]
            xj, yj = ring[j][0], ring[j][1]
            if ((yi > y) != (yj > y)) and (x < (xj - xi) * (y - yi) / ((yj - yi) or 1e-12) + xi):
                inside = not inside
            j = i
        if inside:
            return True
    return False

def sort_by_quality(features, incident_date):
    # Best-first ranking shared by the composite path. Score = (cloud_cover,
    # -sun_elevation, |dt|), exactly the single-scene ranking used previously.
    inc = incident_date
    if inc.tzinfo is not None:
        inc = inc.tz_localize(None)
    def score(f):
        p = f.get('properties', {})
        acq = pd.to_datetime(p.get('acquired'), errors='coerce')
        if pd.notna(acq) and acq.tzinfo is not None:
            acq = acq.tz_localize(None)
        d = abs((acq - inc).total_seconds()) if pd.notna(acq) else float('inf')
        return (p.get('cloud_cover', 1.0), -p.get('sun_elevation', 0.0), d)
    return sorted(features, key=score)

def select_covering_scenes(features, bbox, threshold=COVERAGE_THRESHOLD,
                           min_scenes=MIN_SCENES_PER_ORDER, max_scenes=MAX_SCENES_PER_ORDER,
                           sample=COVERAGE_SAMPLE):
    # Walk the (already best-first ranked) scenes and order a REDUNDANT set for the AOI:
    # - the greedy pass guarantees the whole AOI is geometrically covered (straddling scans),
    # - a minimum scene count guarantees redundancy, so cloud / no-data / clipped pixels in
    #   the best scene can be patched by the other scenes in the downstream UDM-masked mosaic.
    # A scene is only ever skipped once the min-count is met AND it adds no new coverage.
    xs = np.linspace(bbox['min_lon'], bbox['max_lon'], sample)
    ys = np.linspace(bbox['min_lat'], bbox['max_lat'], sample)
    points = [(x, y) for x in xs for y in ys]
    total = len(points)
    covered = set()
    chosen = []
    for f in features:
        if len(chosen) >= max_scenes:
            break
        if len(chosen) >= min_scenes and total and len(covered) / total >= threshold:
            break
        geom = f.get('geometry')
        if not geom:
            continue
        local = {i for i, (x, y) in enumerate(points) if _point_in_geometry(x, y, geom)}
        added = local - covered
        if not added and len(chosen) >= min_scenes:
            continue
        covered |= added
        chosen.append(f)
    coverage = (len(covered) / total) if total else 0.0
    return chosen, coverage

def tiered_search(bbox, incident_date, lat):
    # Returns {'tier_used': int|None, 'scenes': list|None, 'coverage': float, 'sun_max': float,
    #          'straddles_scans': bool}
    # quick-search requires RFC 3339 (UTC) timestamps, e.g. 2024-12-17T00:00:00Z.
    sun_max = theoretical_max_elevation(incident_date, lat)
    for idx, tier in enumerate(FALLBACK_TIERS):
        start = (incident_date + timedelta(days=1)).strftime('%Y-%m-%dT00:00:00Z')
        end   = (incident_date + timedelta(days=tier['window_days'])).strftime('%Y-%m-%dT23:59:59Z')
        min_sun = sun_max * tier['sun_fraction']
        feats = quick_search(bbox, start, end, tier['max_cloud'], min_sun)
        if not feats:
            continue
        ranked = sort_by_quality(feats, incident_date)
        chosen, coverage = select_covering_scenes(ranked, bbox)
        if chosen:
            return {'tier_used': idx, 'scenes': chosen, 'coverage': coverage,
                    'sun_max': sun_max, 'straddles_scans': len(chosen) > 1}
    return {'tier_used': None, 'scenes': None, 'coverage': 0.0,
            'sun_max': sun_max, 'straddles_scans': False}

def submit_order(name, scenes, bbox):
    # Proven Planet orders-v2 payload format (respecting planet_order_working.ipynb).
    # Submits ALL covering scenes together so the (clipped) products can be mosaiced into
    # one complete composite downstream - never a single, possibly-clipped scene.
    item_ids = [s['id'] for s in scenes]
    payload = {
        'name': name,
        'products': [{'item_ids': item_ids, 'item_type': 'PSScene', 'product_bundle': 'analytic_sr_udm2'}],
        'tools': [{'clip': {'aoi': bbox_to_polygon(bbox)}}],
    }
    r = PLANET.post(ORDERS_API, json=payload, timeout=180)
    r.raise_for_status()
    return r.json()['id']




In [ ]:
# STEP 1 - Load the dedup notebook's landslide.csv (already resized + deduped).
# Each row already carries: incident, candidate, label, incident_date, and the
# fixed-size (centroid-preserving) bounding box. Nothing is re-resized here.
cand = pd.read_csv(LANDSLIDE_CSV)[:5]
cand['incident']  = cand['incident'].astype(str).str.strip()
cand['candidate'] = cand['candidate'].astype(int)
for cc in ['min_lat', 'max_lat', 'min_lon', 'max_lon']:
    cand[cc] = pd.to_numeric(cand[cc], errors='coerce')
cand['key'] = cand['incident'] + '_' + cand['candidate'].astype(str)
print('landslide.csv rows:', len(cand))

cand['incident_date'] = pd.to_datetime(cand['incident_date'], errors='coerce')

bad_date = cand['incident_date'].isna()
bad_box  = cand[['min_lat', 'max_lat', 'min_lon', 'max_lon']].isna().any(axis=1)

# Log failures loudly (do not crash): rows with missing dates or invalid coords.
failed = cand[bad_date | bad_box][['key', 'incident', 'candidate']].copy()
failed['reason'] = ''
failed.loc[bad_date, 'reason'] = 'missing_or_invalid_date'
failed.loc[bad_box, 'reason']  = 'missing_or_invalid_coords'
print('--- validation failures (logged & skipped) ---')
print(failed.to_string(index=False))

cand = cand[~(bad_date | bad_box)].copy()
print('usable landslides:', len(cand))


In [ ]:
# STEP 2 - Build order records from the deduped landslide.csv boxes directly.
# The boxes are already fixed-size and deduplicated, so no re-resize / re-dedupe.
records = []
for _, row in cand.iterrows():
    fb = {
        'min_lon': float(row['min_lon']),
        'max_lon': float(row['max_lon']),
        'min_lat': float(row['min_lat']),
        'max_lat': float(row['max_lat']),
    }
    lat_c = (row['min_lat'] + row['max_lat']) / 2.0
    lon_c = (row['min_lon'] + row['max_lon']) / 2.0
    orig_w = (row['max_lon'] - row['min_lon']) * 111000.0 * math.cos(math.radians(lat_c))
    orig_h = (row['max_lat'] - row['min_lat']) * 111000.0
    records.append({
        'key': row['key'], 'incident': row['incident'], 'candidate': int(row['candidate']),
        'incident_date': row['incident_date'],
        'lat_c': lat_c, 'lon_c': lon_c,
        'orig_box_m': {'w_m': round(orig_w, 2), 'h_m': round(orig_h, 2)},
        'final_bbox': fb, 'dup_dropped': False,
    })
print('records after fixed-size resize:', len(records))

order_records = records
print('records eligible for ordering:', len(order_records))


In [ ]:
# STEP 5+6+7 - Search + submit order, guarded shared state, log persistence.
shared_lock = threading.Lock()

def load_remote_json(remote, default):
    try:
        p = hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
                            filename=remote, token=HF_TOKEN)
        with open(p) as f:
            return json.load(f)
    except Exception:
        return default

# Fetch the order log fresh from HF at the start; skip candidates with an existing order_id.
order_log = load_remote_json(ORDER_LOG_REMOTE, {})
if not isinstance(order_log, dict):
    order_log = {}
print('existing order-log entries:', len(order_log))

# Main-thread checkpoint counter (used to gate HF uploads)
new_entries_lock  = threading.Lock()
new_entries_count = 0

def process_order(rec):
    # Reads/writes the global order_log under a lock; runs in worker threads.
    key = rec['key']
    with shared_lock:
        if order_log.get(key) and order_log[key].get('order_id'):
            return ('skipped_existing', key)

    entry = {
        'key': key, 'incident': rec['incident'], 'candidate': rec['candidate'],
        'incident_date': rec['incident_date'].strftime('%Y-%m-%d'),
        'orig_box_m': rec['orig_box_m'],
        'final_bbox': rec['final_bbox'],
        'final_bbox_m': TARGET_BOX_SIZE_M,
        'status': None, 'order_id': None, 'item_id': None,
        'cloud_cover': None, 'sun_elevation': None, 'sun_max': None,
        'sun_required': None, 'fallback_tier_used': None, 'acquired': None,
    }

    res = tiered_search(rec['final_bbox'], rec['incident_date'], rec['lat_c'])
    entry['sun_max'] = round(float(res['sun_max']), 2)

    if res['scenes'] is None:
        entry['status'] = 'no_scene_found_after_all_tiers'
        entry['no_scene_found_after_all_tiers'] = True   # visible + countable failure
    else:
        scenes = res['scenes']
        p = scenes[0]['properties']
        tier = FALLBACK_TIERS[res['tier_used']]
        entry['fallback_tier_used'] = res['tier_used']
        entry['item_id']       = scenes[0]['id']
        entry['item_ids']      = [s['id'] for s in scenes]      # full composite scene set
        entry['scene_count']   = len(scenes)
        entry['aoi_coverage']  = round(float(res['coverage']), 4)  # fraction of AOI the composite covers
        entry['straddles_scans'] = bool(res['straddles_scans'])    # True when >1 scene was needed
        entry['cloud_cover']   = p.get('cloud_cover')
        entry['sun_elevation'] = p.get('sun_elevation')
        entry['acquired']      = p.get('acquired')
        entry['sun_required']  = round(res['sun_max'] * tier['sun_fraction'], 2)
        try:
            name = f'incident_{rec["incident"]}_cand_{rec["candidate"]}'
            entry['order_id'] = submit_order(name, scenes, rec['final_bbox'])
            entry['status'] = 'order_submitted'
            entry['timestamp'] = datetime.now(timezone.utc).isoformat()
        except Exception as e:
            entry['status'] = 'order_failed'
            entry['error'] = str(e)

    with shared_lock:
        order_log[key] = entry
    with new_entries_lock:
        global new_entries_count
        new_entries_count += 1
    return (entry['status'], key)

In [ ]:
def persist_local():
    os.makedirs(ORDER_STATE_DIR, exist_ok=True)
    with shared_lock:
        snap = dict(order_log)            # snapshot under lock to avoid a dict-size-mutation race
    with open(f'{ORDER_STATE_DIR}/order_log.json', 'w') as f:
        json.dump(snap, f, indent=1, default=str)

def checkpoint_main_thread():
    global new_entries_count
    # MAIN thread only. upload_folder -> one HF commit.
    with new_entries_lock:
        n = new_entries_count
        if n <= 0:
            return
    persist_local()
    try:
        upload_folder(folder_path=ORDER_STATE_DIR, path_in_repo='local',
                      repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
                      commit_message=f'order log checkpoint ({n} new)', token=HF_TOKEN)
        with new_entries_lock:
            new_entries_count = 0
        print(f'[main] uploaded order_log.json checkpoint ({len(order_log)} entries)')
    except Exception as e:
        print('[main] checkpoint upload failed (local state kept):', e)

os.makedirs(ORDER_STATE_DIR, exist_ok=True)
persist_local()

outcomes = {}
completed = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_order, rec): rec['key'] for rec in order_records}
    for fut in as_completed(futures):
        try:
            st, k = fut.result()
            outcomes[st] = outcomes.get(st, 0) + 1
        except Exception as e:
            outcomes['worker_error'] = outcomes.get('worker_error', 0) + 1
            print('worker error:', e)
        completed += 1
        if completed % LOG_FLUSH_EVERY == 0:
            checkpoint_main_thread()

checkpoint_main_thread()   # flush any partial batch at the end
persist_local()

print('\n===== submission outcomes =====')
for k, v in sorted(outcomes.items()):
    print(f'  {k}: {v}')

tier_break = {}
for en in order_log.values():
    if en.get('order_id'):
        t = en.get('fallback_tier_used')
        tier_break[t if t is not None else 'none'] = tier_break.get(t if t is not None else 'none', 0) + 1
print('\nfallback tier used per submitted order (0 = ideal, higher = relaxed):')
for t in sorted(tier_break.keys()):
    print(f'  tier {t}: {tier_break[t]}')

print('\nStage 1 complete. Run Stage 2 to download + enrich.')